In [2]:
pip install numpy

   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ---------------------------------------  12.1/12.3 MB 75.1 MB/s eta 0:00:01
   ---------------------------------------- 12.3/12.3 MB 59.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
pip install paho-mqtt

  Using cached paho_mqtt-2.1.0-py3-none-any.whl.metadata (23 kB)
Using cached paho_mqtt-2.1.0-py3-none-any.whl (67 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
pip install matplotlib

  Using cached matplotlib-3.10.8-cp312-cp312-win_amd64.whl.metadata (52 kB)
  Using cached contourpy-1.3.3-cp312-cp312-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached kiwisolver-1.4.9-cp312-cp312-win_amd64.whl.metadata (6.4 kB)
Using cached matplotlib-3.10.8-cp312-cp312-win_amd64.whl (8.1 MB)
Using cached contourpy-1.3.3-cp312-cp312-win_amd64.whl (226 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ---------------------------------------- 2.3/2.3 MB 26.4 MB/s eta 0:00:00
Using cached kiwisolver-1.4.9-cp312-cp312-win_amd64.whl (73 kB)
   ---------------------------------------- 0.0/7.0 MB ? eta -:--:--
   ---------------------------------------- 7.0/7.0 MB 48.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import json
import time
from collections import defaultdict, deque

import paho.mqtt.client as mqtt

# --- CONFIG ---
BROKER_HOST = "127.0.0.1"   # broker is on your PC
BROKER_PORT = 1883
TOPIC       = "ips/rssi"  # match: ips/anchor1/rssi, ips/anchor2/rssi, ...

# number of recent samples to keep per tag (for smoothing)
WINDOW_SIZE = 10

# state: recent RSSI values per (anchor, mac)
rssi_buffer = defaultdict(lambda: deque(maxlen=WINDOW_SIZE))

def compute_avg_rssi(anchor, mac):
    buf = rssi_buffer[(anchor, mac)]
    if not buf:
        return None
    return sum(buf) / len(buf)

# MQTT callbacks
def on_connect(client, userdata, flags, rc):
    print(f"[MQTT] Connected with result code {rc}")
    client.subscribe(TOPIC)
    print(f"[MQTT] Subscribed to {TOPIC}")

def on_message(client, userdata, msg):
    try:
        payload = msg.payload.decode("utf-8")
        data = json.loads(payload)
    except Exception as e:
        print(f"[ERROR] Failed to decode message on {msg.topic}: {e}")
        return

    # Expected JSON from ESP32:
    # {"anchor":"A1","mac":"cb:a1:4f:92:8d:11","rssi":-67}
    anchor = data.get("anchor", "unknown")
    mac    = data.get("mac", "unknown")
    rssi   = data.get("rssi", None)

    if rssi is None:
        return

    # update buffer and compute moving average
    rssi_buffer[(anchor, mac)].append(rssi)
    rssi_avg = compute_avg_rssi(anchor, mac)

    # timestamp for logging
    ts = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())

    print(
        f"{ts} | anchor={anchor:>2} tag={mac} "
        f"rssi={rssi:>4} dBm  avg={rssi_avg:6.2f} dBm"
    )

    # TODO: here you can call your localisation / storage code
    # e.g. write to file, send to DB, run multilateration, etc.


def main():
    client = mqtt.Client(client_id="ips-processor")
    client.on_connect = on_connect
    client.on_message = on_message

    client.connect(BROKER_HOST, BROKER_PORT, keepalive=60)

    print("[MAIN] Starting MQTT loop, press Ctrl+C to stop")
    try:
        client.loop_forever()
    except KeyboardInterrupt:
        print("\n[MAIN] Stopping...")
        client.disconnect()

if __name__ == "__main__":
    main()


C:\Users\wayne\AppData\Local\Temp\ipykernel_34100\2030814585.py:64: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = mqtt.Client(client_id="ips-processor")


[MAIN] Starting MQTT loop, press Ctrl+C to stop
[MQTT] Connected with result code 0
[MQTT] Subscribed to ips/rssi
2026-01-08 17:39:41 | anchor=A1 tag=7b:70:fc:57:ce:ae rssi= -72 dBm  avg=-72.00 dBm
2026-01-08 17:39:42 | anchor=A1 tag=7b:70:fc:57:ce:ae rssi= -72 dBm  avg=-72.00 dBm
2026-01-08 17:39:43 | anchor=A1 tag=7b:70:fc:57:ce:ae rssi= -75 dBm  avg=-73.00 dBm
2026-01-08 17:39:44 | anchor=A1 tag=7b:70:fc:57:ce:ae rssi= -75 dBm  avg=-73.50 dBm
2026-01-08 17:39:45 | anchor=A1 tag=7b:70:fc:57:ce:ae rssi= -74 dBm  avg=-73.60 dBm
2026-01-08 17:39:46 | anchor=A1 tag=7b:70:fc:57:ce:ae rssi= -72 dBm  avg=-73.33 dBm
2026-01-08 17:39:47 | anchor=A1 tag=7b:70:fc:57:ce:ae rssi= -75 dBm  avg=-73.57 dBm
2026-01-08 17:39:48 | anchor=A1 tag=7b:70:fc:57:ce:ae rssi= -73 dBm  avg=-73.50 dBm
2026-01-08 17:39:49 | anchor=A1 tag=7b:70:fc:57:ce:ae rssi= -67 dBm  avg=-72.78 dBm
2026-01-08 17:39:50 | anchor=A1 tag=7b:70:fc:57:ce:ae rssi= -73 dBm  avg=-72.80 dBm
2026-01-08 17:39:51 | anchor=A1 tag=7b:70:fc:5

In [6]:
pip install fastapi uvicorn paho-mqtt


  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
Using cached h11-0.16.0-py3-none-any.whl (37 kB)
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 27.9 MB/s eta 0:00:00
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
Using cached typing_inspection-0.4.2-py3-none-any.whl (14 kB)
Using cached idna-3.11-py3-none-any.whl (71 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
pip install pandas


  Using cached pandas-2.3.3-cp312-cp312-win_amd64.whl.metadata (19 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
Using cached pandas-2.3.3-cp312-cp312-win_amd64.whl (11.0 MB)
Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
pip install scipy

   ---------------------------------------- 0.0/36.3 MB ? eta -:--:--
   - -------------------------------------- 1.0/36.3 MB 6.3 MB/s eta 0:00:06
   -- ------------------------------------- 2.6/36.3 MB 6.9 MB/s eta 0:00:05
   ---- ----------------------------------- 4.5/36.3 MB 7.3 MB/s eta 0:00:05
   ------ --------------------------------- 5.5/36.3 MB 6.8 MB/s eta 0:00:05
   ------- -------------------------------- 7.1/36.3 MB 7.0 MB/s eta 0:00:05
   --------- ------------------------------ 8.7/36.3 MB 7.1 MB/s eta 0:00:04
   ---------- ----------------------------- 10.0/36.3 MB 7.0 MB/s eta 0:00:04
   ------------ --------------------------- 11.5/36.3 MB 7.0 MB/s eta 0:00:04
   ------------- -------------------------- 12.6/36.3 MB 6.7 MB/s eta 0:00:04
   --------------- ------------------------ 13.6/36.3 MB 6.6 MB/s eta 0:00:04
   ---------------- ----------------------- 14.9/36.3 MB 6.6 MB/s eta 0:00:04
   ----------------- ---------------------- 16.3/36.3 MB 6.6 MB/s eta 0:00:04



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip
